In [ ]:
# Install once at the start of Run All.
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve("https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/5971c8ff87a230ac0d68c1258b575995cfdb17c0/kaggle/cube555_github/setup.py", setup_dir + "/setup.py")
    run_path(setup_file)


In [ ]:
import cayleypy as cp
import multigpubeamsearch as beam
from multigpubeamsearch import cube555

# Puzzle
PUZZLE_ID_START = 1020
PUZZLE_ID_END = 1020
INDICES_URL = None
RANDOM_PUZZLE = False
START_STATE_OVERRIDE = None  # Optional; use with one puzzle

# Search
BEAM_WIDTH = 1_048_576  # Integer or "max"
MAX_STEPS = 140
BEAM_MODE = "simple"
USE_CHILD_SCORES = True
RETURN_PATH = True
NUM_GPUS = None  # Use every visible GPU

# Models: weight 1 = Transformer, 0 = MLP, 0.8 = 80/20 blend
MODEL_ROOT = None
COMPETITION_ROOT = None
CHECKPOINT_PATH = "q555_f1_bell2k.pt"
RESMLP_CHECKPOINT_PATH = "q555_2k_BEST.pt"
LAYOUT_PATH = "piece_layout_555.json"
TRANSFORMER_WEIGHT = 0.8

# Collect, BFS touch, and reflection
SOLUTION_MODE = "collect"
COLLECT_EXTRA_DEPTHS = 1
MAX_COLLECTED_SOLUTIONS = 2_000
TOUCH_BFS_RADIUS = 5
TOUCH_BFS_MAX_ENTRIES = 30_000_000
REFLECT_MODE = "off"  # "off", "after_original", or "only"
REFLECT_SOURCE_CSV = None

# Run and results
RUN_TIMEOUT_SECONDS = 12 * 3600
OUTPUT_ROOT = "cube555_runs"
PUBLISH_RESULTS = True
AUTHOR = "trydotatwo"
KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = 176


# Cube555 blend with CayleyPy

Edit the first visible cell and choose **Run All**. The library reuses a matching saved profile or tunes one for the requested beam. Results are replayed, saved, and published automatically when running on Kaggle with two GPUs.


In [ ]:
puzzle_ids = cube555.select_puzzles(PUZZLE_ID_START, PUZZLE_ID_END,
    indices_url=INDICES_URL, random_choice=RANDOM_PUZZLE)
puzzle = cube555.load(puzzle_ids[0], model_root=MODEL_ROOT,
    competition_root=COMPETITION_ROOT)
graph = cp.CayleyGraph(puzzle.definition, device="cpu", random_seed=42)
predictor = puzzle.blend(graph, transformer_weight=TRANSFORMER_WEIGHT,
    transformer_checkpoint=CHECKPOINT_PATH, mlp_checkpoint=RESMLP_CHECKPOINT_PATH,
    layout_path=LAYOUT_PATH)
beam.enable_native(beam.NativeOptions(num_gpus=NUM_GPUS,
    timeout_seconds=RUN_TIMEOUT_SECONDS))


In [ ]:
with beam.NativeSession():
    for puzzle_id in puzzle_ids:
        puzzle = cube555.load(puzzle_id, model_root=MODEL_ROOT,
            competition_root=COMPETITION_ROOT)
        start_state = START_STATE_OVERRIDE or puzzle.start_state
        result = graph.beam_search(
            start_state=start_state, predictor=predictor, backend="native",
            beam_width=BEAM_WIDTH, max_steps=MAX_STEPS, beam_mode=BEAM_MODE,
            use_child_scores=USE_CHILD_SCORES, return_path=RETURN_PATH,
            solution_mode=SOLUTION_MODE, collect_extra_depths=COLLECT_EXTRA_DEPTHS,
            max_collected_solutions=MAX_COLLECTED_SOLUTIONS,
            bfs_radius=TOUCH_BFS_RADIUS, bfs_max_entries=TOUCH_BFS_MAX_ENTRIES,
            reflect_mode=REFLECT_MODE,
            reflection_source=cube555.source_solution(REFLECT_SOURCE_CSV, puzzle_id, graph),
        )
        saved = cube555.finish_result(result, graph, start_state,
            publish=PUBLISH_RESULTS, directory=OUTPUT_ROOT, puzzle_id=puzzle_id,
            predictor=predictor, author=AUTHOR, kaggle_owner=KAGGLE_OWNER,
            kaggle_slug=KAGGLE_SLUG, kaggle_version=KAGGLE_VERSION)["saved"]
        print(puzzle_id, "best length:", result.path_length,
              "solutions:", len(result.solutions), "saved:", saved)
